# 植物图像检索查询系统 (CLIP ViT-B/32)

**作者**: Zihan Yin

**项目概述**: 本项目是植物图像检索的查询端，使用已建立的CLIP索引对用户上传的植物图像进行相似度检索，返回最匹配的植物种类Top-K结果。

## 步骤 1 — 导入库和路径配置

**目的**: 导入必要的Python库，配置索引文件、模型文件的路径。

**关键路径**:
| 路径 | 说明 |
|------|------|
| INDEX_PATH | 索引嵌入向量文件（由索引脚本生成） |
| META_PATH | 元数据JSON文件（由索引脚本生成） |
| MODEL_DIR | 本地CLIP模型缓存目录 |
| MODEL_ID | 在线模型标识（本地加载失败时的备选） |

**设备检测**: 自动检测并使用GPU（CUDA），若无则回退到CPU。

In [ ]:
# ========== 导入库 ==========
from pathlib import Path                          # 路径处理
import json                                       # JSON解析
import numpy as np                                # 数值计算
from PIL import Image, ImageOps                   # 图像处理

import torch                                      # PyTorch框架
from torch import nn                              # 神经网络模块
from torchvision import transforms                # 图像变换/增强
from transformers import CLIPModel, CLIPProcessor # CLIP模型和处理器

# ========== 路径配置 ==========
INDEX_PATH = Path("index/embeddings_fp16.npz")    # 索引向量文件（索引脚本生成）
META_PATH  = Path("index/meta.json")              # 元数据文件（索引脚本生成）
MODEL_DIR  = Path("clip-vit-b32")                 # 本地CLIP模型目录
MODEL_ID   = "openai/clip-vit-base-patch32"       # 在线模型标识（本地失败时的备选）
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"  # 检测设备（优先GPU）

print("设备:", DEVICE)

Device: cuda


## 步骤 2 — 工具函数（图像预处理 + TTA）

**目的**: 定义图像预处理和测试时增强（TTA）函数。

**两个核心函数**:

1. **`exif_correct()`**: 
   - 修正照片EXIF方向信息
   - 将RGBA转换为RGB（白色背景）
   - 与索引阶段的预处理保持一致

2. **`tta_pipeline()`**: 
   - 轻度测试时增强（Test-Time Augmentation）
   - 包括：水平翻转、小幅仿射变换、轻微颜色抖动
   - 目的：提高查询鲁棒性，避免单次预处理偏差

**注意**: TTA强度比索引阶段更轻，避免引入过多噪声。

In [ ]:
# ========== 工具函数（图像预处理 + TTA） ==========
def exif_correct(img: Image.Image) -> Image.Image:
    """修正EXIF方向信息并转换为RGB格式"""
    img = ImageOps.exif_transpose(img)            # 根据EXIF自动旋转
    if img.mode == "RGBA":                        # 处理透明通道
        bg = Image.new("RGB", img.size, (255, 255, 255))  # 白色背景
        bg.paste(img, mask=img.split()[-1])       # 使用alpha通道作为掩码
        return bg
    if img.mode != "RGB":                         # 确保为RGB格式
        img = img.convert("RGB")
    return img


def tta_pipeline():
    """轻度测试时增强（避免过强的变换）"""
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),                      # 随机水平翻转
        transforms.RandomAffine(degrees=10, translate=(0.06, 0.06),  # 小幅仿射变换
                                scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1,         # 轻微颜色抖动
                               saturation=0.1, hue=0.01),
    ])

## 步骤 3 — 加载索引和CLIP模型

**目的**: 加载已保存的嵌入向量索引和CLIP模型，准备进行检索。

**加载内容**:
1. **索引文件** (`embeddings_fp16.npz`):
   - 嵌入向量矩阵 (N, D)
   - 植物ID数组

2. **元数据文件** (`meta.json`):
   - 模型信息、维度、类别数等

3. **CLIP模型**:
   - 优先从本地缓存加载
   - 失败时回退到在线下载（Hugging Face）

**模型设置**: 评估模式 (`eval()`)，移动到指定设备。

In [ ]:
# ========== 加载索引和CLIP模型 ==========
# 加载索引文件
npz = np.load(INDEX_PATH)
db_emb    = npz["embeddings"].astype(np.float32)  # 数据库嵌入向量 (N, D)
plant_ids = npz["plant_ids"]                      # 植物ID数组

with open(META_PATH, "r", encoding="utf-8") as f:  # 加载元数据
    meta = json.load(f)

print("索引中的类别数量:", db_emb.shape[0])
print("嵌入向量维度:", db_emb.shape[1])

# 加载CLIP模型（优先本地，失败则在线下载）
try:
    model     = CLIPModel.from_pretrained(MODEL_DIR)     # 从本地加载模型
    processor = CLIPProcessor.from_pretrained(MODEL_DIR) # 从本地加载处理器
    print("本地模型加载成功。")
except Exception:
    print("本地模型加载失败，回退到在线下载。")
    model     = CLIPModel.from_pretrained(MODEL_ID)      # 从Hugging Face在线加载
    processor = CLIPProcessor.from_pretrained(MODEL_ID)

model.eval().to(DEVICE)                                  # 设置评估模式并移动到设备

Number of classes in index: 1319
Embedding dimension: 512
Failed to load local model; falling back to online.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

## 步骤 4 — 编码查询图像（用户上传）

**目的**: 将用户上传的查询图像转换为嵌入向量，使用TTA提高鲁棒性。

**处理流程**:
1. EXIF修正和格式转换
2. 生成TTA增强样本（原始图 + 8个增强版本）
3. 使用CLIP模型批量编码
4. L2归一化
5. 计算所有增强样本的平均向量作为最终查询向量

**函数参数**:
- `img`: PIL图像对象
- `tta_num`: TTA增强样本数量（默认8）

**输出**: 归一化的查询嵌入向量 (D,)

In [ ]:
# ========== 编码查询图像（用户上传） ==========
@torch.no_grad()  # 禁用梯度计算，节省内存
def encode_query(img: Image.Image, tta_num=8):
    """
    对上传图像进行TTA增强编码，返回单个查询向量
    
    参数:
        img: PIL图像对象
        tta_num: TTA增强样本数量（默认8）
    
    返回:
        归一化的查询嵌入向量 (D,)
    """
    img = exif_correct(img)                              # EXIF修正
    tta = tta_pipeline()                                 # 初始化TTA管道
    pil_list = [img] + [tta(img) for _ in range(max(tta_num-1, 0))]  # 原始图+增强图

    # 使用CLIP处理器预处理
    inputs = processor(images=pil_list, return_tensors="pt",
                       do_center_crop=True, padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}  # 移动到设备

    # 混合精度推理（GPU加速）
    with torch.autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"),
                        dtype=torch.float16, enabled=(DEVICE=="cuda")):
        feats = model.get_image_features(**inputs)         # 获取图像特征

    feats = nn.functional.normalize(feats, p=2, dim=-1)    # L2归一化
    q = feats.mean(dim=0, keepdim=False)                   # 计算平均向量 (D,)
    return q.cpu().numpy().astype(np.float32)              # 转换为NumPy数组

## 步骤 5 — 计算余弦相似度并获取Top-K

**目的**: 计算查询向量与索引中所有原型的余弦相似度，返回最匹配的Top-K结果。

**算法说明**:
- 由于索引向量已L2归一化，余弦相似度 = 向量点积
- 使用 `np.argpartition` 高效获取Top-K索引（比全排序更快）
- 返回格式：(植物ID, 相似度分数) 列表

**参数**:
- `query_emb`: 查询向量 (D,)
- `db_emb`: 数据库嵌入矩阵 (N, D)
- `plant_ids`: 植物ID数组
- `k`: 返回结果数量（默认10）

In [ ]:
# ========== 计算余弦相似度并获取Top-K ==========
def topk_cosine(query_emb, db_emb, plant_ids, k=10):
    """
    计算余弦相似度并返回Top-K匹配结果
    
    参数:
        query_emb: 查询向量 (D,)
        db_emb: 数据库嵌入矩阵 (N, D)，已L2归一化
        plant_ids: 植物ID数组
        k: 返回结果数量（默认10）
    
    返回:
        Top-K结果列表 [(plant_id, score), ...]
    """
    sims = db_emb @ query_emb                              # 余弦相似度（点积）
    idx = np.argpartition(-sims, kth=min(k, len(sims)-1))[:k]  # 高效获取Top-K索引
    idx = idx[np.argsort(-sims[idx])]                      # 按相似度降序排序
    return [(int(plant_ids[i]), float(sims[i])) for i in idx]  # 返回(植物ID, 分数)

## 步骤 6 — 示例查询

**目的**: 演示如何使用上述函数对单张图像进行检索查询。

**使用说明**:
1. 将 `QUERY_IMG` 替换为您要查询的图像路径
2. 支持的图像格式：`.jpg`, `.webp`, `.JPG`, `.PNG` 等
3. 运行后输出Top-10匹配结果及相似度分数

**输出格式**: `plant_id=数字 score=0.xxxx`  
**分数解读**: 分数越接近1.0表示相似度越高。

In [ ]:
# ========== 示例查询 ==========
# 支持的图像后缀：.jpg, .webp, .JPG, .PNG等

# 替换为您要查询的图像路径
QUERY_IMG = Path("test_images/fdghjmhgferghthjkhgf.jpg")  

img = Image.open(QUERY_IMG)                                # 打开查询图像
q_emb = encode_query(img, tta_num=8)                       # 编码查询向量

results = topk_cosine(q_emb, db_emb, plant_ids, k=10)      # 获取Top-10结果
for pid, score in results:
    print(f"plant_id={pid}\tscore={score:.4f}")            # 输出结果

plant_id=342	score=0.8903
plant_id=343	score=0.8866
plant_id=341	score=0.8801
plant_id=250	score=0.8756
plant_id=540	score=0.8733
plant_id=340	score=0.8713
plant_id=402	score=0.8682
plant_id=348	score=0.8681
plant_id=742	score=0.8677
plant_id=236	score=0.8664
